In [ ]:
# NOTEBOOK NAME
# PPImasker.ipynb
# NOTEBOOK NAME

# # OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

from pathlib import Path      # used to play with pathnames to save

# # SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions')
from CustomFunctions1 import *

import dask
from dask import delayed
from dask.distributed import Client, LocalCluster, progress

# for live progress bar
from dask.distributed import as_completed
from tqdm.notebook import tqdm

In [ ]:
# Data Masking Section
# Original Version without Dask

import warnings # !!!!!
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0', category=RuntimeWarning)


# set your Quality Control Options:
# 0 for no quality control
QualityControlOption = 2

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Brisbane)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
# RadarDay   = 14 # looped down below

# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '00:00'
LoopEndTime   = '23:55'

if (QualityControlOption == 0):
    VarianceGridSize = '3x3'
    MinZ             = 'none' # [dBZ]                           # write 'none' [string] for no limit
    MinRhoHV         = 'none' # [0 to 1 correlation]            # write 'none' [string] for no limit
    
    MaxZvariance     = 'none' # [dBZ^2]                         # write 'none' [string] for no limit
    MaxVvariance     = 'none' # [m^2/s^2]                       # write 'none' [string] for no limit
    MaxZDRvariance   = 'none' # [dB^2]                          # write 'none' [string] for no limit
    MaxRhoHVvariance = 'none' # [0 to 1 correlation squared]    # write 'none' [string] for no limit

    # but maybe you only want to consider those cells with at least a few neighbours!
    MinZcount        =  3     # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinVcount        = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinZDRcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinRhoHVcount    = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinKDPcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    
elif (QualityControlOption == 1):
    VarianceGridSize = '3x3'
    MinZ             =  5     # [dBZ]                           # write 'none' [string] for no limit
    MinRhoHV         =  0.88  # [0 to 1 correlation]            # write 'none' [string] for no limit
    
    MaxZvariance     =  15    # [dBZ^2]                         # write 'none' [string] for no limit
    MaxVvariance     = 'none' # [m^2/s^2]                       # write 'none' [string] for no limit
    MaxZDRvariance   = 'none' # [dB^2]                          # write 'none' [string] for no limit
    MaxRhoHVvariance = 'none' # [0 to 1 correlation squared]    # write 'none' [string] for no limit

    # but maybe you only want to consider those cells with at least a few neighbours!
    MinZcount        =  3     # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinVcount        = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinZDRcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinRhoHVcount    = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinKDPcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    
elif (QualityControlOption == 2):
    VarianceGridSize = '3x3'
    MinZ             =  0     # [dBZ]                           # write 'none' [string] for no limit
    MinRhoHV         =  0.88  # [0 to 1 correlation]            # write 'none' [string] for no limit
    
    MaxZvariance     =  15    # [dBZ^2]                         # write 'none' [string] for no limit
    MaxVvariance     = 'none' # [m^2/s^2]                       # write 'none' [string] for no limit
    MaxZDRvariance   = 'none' # [dB^2]                          # write 'none' [string] for no limit
    MaxRhoHVvariance = 'none' # [0 to 1 correlation squared]    # write 'none' [string] for no limit

    # but maybe you only want to consider those cells with at least a few neighbours!
    MinZcount        =  2     # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinVcount        = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinZDRcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinRhoHVcount    = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinKDPcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
# elif (QualityControlOption == 3):
#     VarianceGridSize = '3x3'
#     MinZ             =  0     # [dBZ]                           # write 'none' [string] for no limit
#     MinRhoHV         =  0.88  # [0 to 1 correlation]            # write 'none' [string] for no limit
    
#     MaxZvariance     =  15    # [dBZ^2]                         # write 'none' [string] for no limit
#     MaxVvariance     = 'none' # [m^2/s^2]                       # write 'none' [string] for no limit
#     MaxZDRvariance   = 'none' # [dB^2]                          # write 'none' [string] for no limit
#     MaxRhoHVvariance = 'none' # [0 to 1 correlation squared]    # write 'none' [string] for no limit

#     # but maybe you only want to consider those cells with at least a few neighbours!
#     MinZcount        =  2     # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
#     MinVcount        = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
#     MinZDRcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
#     MinRhoHVcount    = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
#     MinKDPcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
# elif (QualityControlOption == 4):
    # VarianceGridSize = '3x3'
    # MinZ             =  0     # [dBZ]                           # write 'none' [string] for no limit
    # MinRhoHV         =  0.88  # [0 to 1 correlation]            # write 'none' [string] for no limit
    
    # MaxZvariance     =  15    # [dBZ^2]                         # write 'none' [string] for no limit
    # MaxVvariance     = 'none' # [m^2/s^2]                       # write 'none' [string] for no limit
    # MaxZDRvariance   = 'none' # [dB^2]                          # write 'none' [string] for no limit
    # MaxRhoHVvariance = 'none' # [0 to 1 correlation squared]    # write 'none' [string] for no limit

    # # but maybe you only want to consider those cells with at least a few neighbours!
    # MinZcount        =  2     # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    # MinVcount        = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    # MinZDRcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    # MinRhoHVcount    = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    # MinKDPcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
else:
    print('Please choose QualityControlOption = 0 or 1 or 2') # or 3 or 4


# loop over all days in February
for RadarDay in range(1,28+1):

    # date choice follow-on
    # add leading zeros for strings
    YYYY = str(RadarYear).zfill(4)
    MM = str(RadarMonth).zfill(2)
    DD = str(RadarDay).zfill(2)
    # write out the data in one string with and without dashes
    RadarFileDate  = YYYY + MM + DD
    RadarFileDatePrint = YYYY + '-' + MM + '-' + DD
    
    
    
    # LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
    # Parse start and end times
    StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
    EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])
    
    # Convert to total minutes for easy comparison
    StartMinOfDay = StartHour * 60 + StartMin
    EndMinOfDay = EndHour * 60 + EndMin
    # LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
    # Parse start and end times
    StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
    EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])
    
    # Convert to total minutes for easy comparison
    StartMinOfDay = StartHour * 60 + StartMin
    EndMinOfDay = EndHour * 60 + EndMin
    
    for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
        # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
        houri = MinOfDay // 60
        mini  = MinOfDay % 60
            
        RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
        # add a string of format hh:mm:ss for printing
        RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
        
        print('working on ' + RadarFileDatePrint + ' ' + RadarFileTimePrint)
        
        NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_ppi' + '/'
        NetCDFstorageFile = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi.nc'
        NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile
        
        
        # try to load in the netcdf file and if it doesn't work, just keep going
        try:
            RadarXR = xr.open_dataset(NetCDFstoragePath, decode_timedelta = False) # add the decode_timedelta to shut up a warning
        except FileNotFoundError:
            print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
            continue
    
    
        if (VarianceGridSize == '3x3'):
            if (MaxZvariance != 'none'):
                Zvar, Zcount = GridStatsFast(RadarXR, 'corrected_reflectivity', 3, fill_value=-32.0)                                   # add variances and counts for Z
            if (MaxVvariance != 'none'):
                Vvar, Vcount = GridStatsFast(RadarXR, 'corrected_velocity',     3, fill_value=[-300, 0.009155552842798897])            # add variances and counts for velocity 
            if (MaxRhoHVvariance != 'none'):
                RhoHVvar, RhoHVcount = GridStatsFast(RadarXR, 'corrected_cross_correlation_ratio', 3, fill_value=[0.0])                # add variances and counts for RhoHV # very specific near 0 value keeps appearing, likely not valid data                                                     
            if (MaxZDRvariance != 'none'):
                ZDRvar, ZDRcount = GridStatsFast(RadarXR, 'corrected_differential_reflectivity', 3, fill_value=[-15.0, 10.00015259254738]) # add variances and counts for ZDR # very specific near +10 value keeps appearing, likely not valid data
        elif (VarianceGridSize == '5x5'):
            if (MaxZvariance != 'none'):
                Zvar, Zcount = GridStatsFast(RadarXR, 'corrected_reflectivity', 5, fill_value=-32.0)                                   # add variances and counts for Z
            if (MaxVvariance != 'none'):
                Vvar, Vcount = GridStatsFast(RadarXR, 'corrected_velocity',     5, fill_value=[-300, 0.009155552842798897])            # add variances and counts for velocity 
            if (MaxRhoHVvariance != 'none'):
                RhoHVvar, RhoHVcount = GridStatsFast(RadarXR, 'corrected_cross_correlation_ratio', 5, fill_value=[0.0])                # add variances and counts for RhoHV # very specific near 0 value keeps appearing, likely not valid data                                                     
            if (MaxZDRvariance != 'none'):
                ZDRvar, ZDRcount = GridStatsFast(RadarXR, 'corrected_differential_reflectivity', 5, fill_value=[-15.0, 10.00015259254738]) # add variances and counts for ZDR # very specific near +10 value keeps appearing, likely not valid data
        else:
            print("VarianceGridSize must be exactly '3x3' or '5x5'")
    
    
        # MASK CREATION
        # --- Initialise mask as all True (keep all points) ---
        # Use a known (time, range) variable to get the right shape/coords
        ReferenceVar = RadarXR['corrected_reflectivity']
        QCmask = xr.ones_like(ReferenceVar, dtype=bool)  # True = keep
        
        # --- Simple threshold conditions ---
        if MinZ != 'none':
            QCmask = QCmask & (RadarXR['corrected_reflectivity'] >= MinZ)
        if MinRhoHV != 'none':
            QCmask = QCmask & (RadarXR['corrected_cross_correlation_ratio'] >= MinRhoHV)
        
        # --- Variance conditions ---
        if MaxZvariance != 'none':
            QCmask = QCmask & (Zvar <= MaxZvariance)
        if MaxVvariance != 'none':
            QCmask = QCmask & (Vvar <= MaxVvariance)
        if MaxZDRvariance != 'none':
            QCmask = QCmask & (ZDRvar <= MaxZDRvariance)
        if MaxRhoHVvariance != 'none':
            QCmask = QCmask & (RhoHVvar <= MaxRhoHVvariance)
        
        # --- Count conditions (minimum valid neighbours) ---
        if MinZcount != 'none':
            QCmask = QCmask & (Zcount >= MinZcount)
        if MinVcount != 'none':
            QCmask = QCmask & (Vcount >= MinVcount)
        if MinZDRcount != 'none':
            QCmask = QCmask & (ZDRcount >= MinZDRcount)
        if MinRhoHVcount != 'none':
            QCmask = QCmask & (RhoHVcount >= MinRhoHVcount)
        if MinKDPcount != 'none':
            QCmask = QCmask & (KDPcount >= MinKDPcount)
    
        
        # APPPLY THE GRAND MASK
        # Build a new dataset, masking only variables with exactly (time, range) dims
        MaskedVars = {}
        for VarName, Da in RadarXR.data_vars.items():
            if Da.dims == ('time', 'range'):
                # Where mask is False, replace with NaN
                MaskedVars[VarName] = Da.where(QCmask)
            else:
                # Keep non-(time, range) variables completely untouched
                MaskedVars[VarName] = Da
    
        RadarXR_QC = RadarXR.assign(MaskedVars)
    
        SaveFolder = NetCDFstorageFolder + 'QC' + str(QualityControlOption) + '/'
        SaveFile = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppiQC' + str(QualityControlOption) + '.nc'
        SavePath = SaveFolder + SaveFile
    
        # CHAD EDITTED SAVING SECTION WHICH DELETES AND RE-WRITES SAVED FILES
        # Ensure directory exists with proper permissions
        if not Path(SaveFolder).exists():
            Path(SaveFolder).mkdir(parents=True, exist_ok=True)
        
        # Remove existing file to avoid permission conflicts
        if Path(SavePath).exists():
            Path(SavePath).unlink()
        
        # Write with explicit mode
        try:
            RadarXR_QC.to_netcdf(
                SavePath, 
                mode='w',
                encoding={var: {'zlib': True, 'complevel': 4} for var in RadarXR.data_vars}
            )
            print(f"Successfully saved: {SavePath}")
        except PermissionError as e:
            print(f"Permission denied writing to {SavePath}")
            print(f"Error: {e}")


In [ ]:
# FUNCTION
# CHAD PARALLELISED VERSION

def QCMaskRadarData(
    RadarDay, MinOfDay,
    RadarIDno, RadarYear, RadarMonth,
    LoopStartTime, LoopEndTime,
    QualityControlOption,
    VarianceGridSize,
    MinZ, MinRhoHV,
    MaxZvariance, MaxVvariance, MaxZDRvariance, MaxRhoHVvariance,
    MinZcount, MinVcount, MinZDRcount, MinRhoHVcount, MinKDPcount
):

    import warnings # !!!!!
    warnings.filterwarnings('ignore', message='Degrees of freedom <= 0', category=RuntimeWarning)

    YYYY = str(RadarYear).zfill(4)
    MM   = str(RadarMonth).zfill(2)
    DD   = str(RadarDay).zfill(2)

    RadarFileDate      = YYYY + MM + DD
    RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

    houri = MinOfDay // 60
    mini  = MinOfDay % 60
    RadarFileTime      = str(houri).zfill(2) + str(mini).zfill(2) + '00'
    RadarFileTimePrint = RadarFileTime[0:2] + ':' + RadarFileTime[2:4] + ':' + RadarFileTime[4:6]

    # print(f'working on {RadarFileDatePrint} {RadarFileTimePrint}')

    NetCDFstorageFolder = (
        f'/scratch/v46/sg3241/tmp/UnzippedRadarFiles/'
        f'{RadarIDno}/{RadarIDno}_{RadarFileDate}_ppi/'
    )
    NetCDFstorageFile = f'{RadarIDno}_{RadarFileDate}_{RadarFileTime}_ppi.nc'
    NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile

    try:
        RadarXR = xr.open_dataset(NetCDFstoragePath, decode_timedelta=False)
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        return None

    # --- Variance computation ---
    grid_size = 3 if VarianceGridSize == '3x3' else 5

    Zvar = Zcount = Vvar = Vcount = None
    RhoHVvar = RhoHVcount = ZDRvar = ZDRcount = None

    if MaxZvariance != 'none':
        Zvar, Zcount = GridStatsFast(RadarXR, 'corrected_reflectivity', grid_size, fill_value=-32.0)
    if MaxVvariance != 'none':
        Vvar, Vcount = GridStatsFast(RadarXR, 'corrected_velocity', grid_size, fill_value=[-300, 0.009155552842798897])
    if MaxRhoHVvariance != 'none':
        RhoHVvar, RhoHVcount = GridStatsFast(RadarXR, 'corrected_cross_correlation_ratio', grid_size, fill_value=[0.0])
    if MaxZDRvariance != 'none':
        ZDRvar, ZDRcount = GridStatsFast(RadarXR, 'corrected_differential_reflectivity', grid_size, fill_value=[-15.0, 10.00015259254738])

    # --- Mask creation ---
    ReferenceVar = RadarXR['corrected_reflectivity']
    QCmask = xr.ones_like(ReferenceVar, dtype=bool)

    if MinZ     != 'none': QCmask = QCmask & (RadarXR['corrected_reflectivity']             >= MinZ)
    if MinRhoHV != 'none': QCmask = QCmask & (RadarXR['corrected_cross_correlation_ratio']  >= MinRhoHV)

    if MaxZvariance    != 'none': QCmask = QCmask & (Zvar    <= MaxZvariance)
    if MaxVvariance    != 'none': QCmask = QCmask & (Vvar    <= MaxVvariance)
    if MaxZDRvariance  != 'none': QCmask = QCmask & (ZDRvar  <= MaxZDRvariance)
    if MaxRhoHVvariance!= 'none': QCmask = QCmask & (RhoHVvar<= MaxRhoHVvariance)

    if MinZcount    != 'none': QCmask = QCmask & (Zcount    >= MinZcount)
    if MinVcount    != 'none': QCmask = QCmask & (Vcount    >= MinVcount)
    if MinZDRcount  != 'none': QCmask = QCmask & (ZDRcount  >= MinZDRcount)
    if MinRhoHVcount!= 'none': QCmask = QCmask & (RhoHVcount>= MinRhoHVcount)

    # --- Apply mask ---
    MaskedVars = {}
    for VarName, Da in RadarXR.data_vars.items():
        MaskedVars[VarName] = Da.where(QCmask) if Da.dims == ('time', 'range') else Da

    RadarXR_QC = RadarXR.assign(MaskedVars)

    SaveFolder = f'/scratch/v46/sg3241/tmp/UnzippedRadarFiles/ppi/{RadarIDno}/{RadarFileDate}/QC{QualityControlOption}/'

    SaveFile   = f'{RadarIDno}_{RadarFileDate}_{RadarFileTime}_ppi_QC{QualityControlOption}.nc'
    SavePath   = SaveFolder + SaveFile

    Path(SaveFolder).mkdir(parents=True, exist_ok=True)
    if Path(SavePath).exists():
        Path(SavePath).unlink()

    try:
        RadarXR_QC.to_netcdf(SavePath, mode='w', encoding={var: {'zlib': True, 'complevel': 4} for var in RadarXR.data_vars})
        
        print(f'Successfully saved: {SavePath}')
    except PermissionError as e:
        print(f'Permission denied writing to {SavePath}: {e}')

    return SavePath
    

In [ ]:
# DATA CHOICE SECTION

# set your Quality Control Options:
# 0 for no quality control
QualityControlOption = 2

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Brisbane)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
# RadarDay   = 14 # looped down below

# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '00:00'
LoopEndTime   = '23:55'

if (QualityControlOption == 0):
    VarianceGridSize = '3x3'
    MinZ             = 'none' # [dBZ]                           # write 'none' [string] for no limit
    MinRhoHV         = 'none' # [0 to 1 correlation]            # write 'none' [string] for no limit
    
    MaxZvariance     = 'none' # [dBZ^2]                         # write 'none' [string] for no limit
    MaxVvariance     = 'none' # [m^2/s^2]                       # write 'none' [string] for no limit
    MaxZDRvariance   = 'none' # [dB^2]                          # write 'none' [string] for no limit
    MaxRhoHVvariance = 'none' # [0 to 1 correlation squared]    # write 'none' [string] for no limit

    # but maybe you only want to consider those cells with at least a few neighbours!
    MinZcount        =  3     # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinVcount        = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinZDRcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinRhoHVcount    = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinKDPcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    
elif (QualityControlOption == 1):
    VarianceGridSize = '3x3'
    MinZ             =  5     # [dBZ]                           # write 'none' [string] for no limit
    MinRhoHV         =  0.88  # [0 to 1 correlation]            # write 'none' [string] for no limit
    
    MaxZvariance     =  15    # [dBZ^2]                         # write 'none' [string] for no limit
    MaxVvariance     = 'none' # [m^2/s^2]                       # write 'none' [string] for no limit
    MaxZDRvariance   = 'none' # [dB^2]                          # write 'none' [string] for no limit
    MaxRhoHVvariance = 'none' # [0 to 1 correlation squared]    # write 'none' [string] for no limit

    # but maybe you only want to consider those cells with at least a few neighbours!
    MinZcount        =  3     # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinVcount        = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinZDRcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinRhoHVcount    = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinKDPcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    
elif (QualityControlOption == 2):
    VarianceGridSize = '3x3'
    MinZ             =  0     # [dBZ]                           # write 'none' [string] for no limit
    MinRhoHV         =  0.88  # [0 to 1 correlation]            # write 'none' [string] for no limit
    
    MaxZvariance     =  15    # [dBZ^2]                         # write 'none' [string] for no limit
    MaxVvariance     = 'none' # [m^2/s^2]                       # write 'none' [string] for no limit
    MaxZDRvariance   = 'none' # [dB^2]                          # write 'none' [string] for no limit
    MaxRhoHVvariance = 'none' # [0 to 1 correlation squared]    # write 'none' [string] for no limit

    # but maybe you only want to consider those cells with at least a few neighbours!
    MinZcount        =  2     # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinVcount        = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinZDRcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinRhoHVcount    = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    MinKDPcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
# elif (QualityControlOption == 3):
#     VarianceGridSize = '3x3'
#     MinZ             =  0     # [dBZ]                           # write 'none' [string] for no limit
#     MinRhoHV         =  0.88  # [0 to 1 correlation]            # write 'none' [string] for no limit
    
#     MaxZvariance     =  15    # [dBZ^2]                         # write 'none' [string] for no limit
#     MaxVvariance     = 'none' # [m^2/s^2]                       # write 'none' [string] for no limit
#     MaxZDRvariance   = 'none' # [dB^2]                          # write 'none' [string] for no limit
#     MaxRhoHVvariance = 'none' # [0 to 1 correlation squared]    # write 'none' [string] for no limit

#     # but maybe you only want to consider those cells with at least a few neighbours!
#     MinZcount        =  2     # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
#     MinVcount        = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
#     MinZDRcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
#     MinRhoHVcount    = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
#     MinKDPcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
# elif (QualityControlOption == 4):
    # VarianceGridSize = '3x3'
    # MinZ             =  0     # [dBZ]                           # write 'none' [string] for no limit
    # MinRhoHV         =  0.88  # [0 to 1 correlation]            # write 'none' [string] for no limit
    
    # MaxZvariance     =  15    # [dBZ^2]                         # write 'none' [string] for no limit
    # MaxVvariance     = 'none' # [m^2/s^2]                       # write 'none' [string] for no limit
    # MaxZDRvariance   = 'none' # [dB^2]                          # write 'none' [string] for no limit
    # MaxRhoHVvariance = 'none' # [0 to 1 correlation squared]    # write 'none' [string] for no limit

    # # but maybe you only want to consider those cells with at least a few neighbours!
    # MinZcount        =  2     # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    # MinVcount        = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    # MinZDRcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    # MinRhoHVcount    = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
    # MinKDPcount      = 'none' # [0 to 9 [int] for 3x3 grids, 0 to 25 for 5x5 grids]   # write 'none' [string] for no limit
else:
    print('Please choose QualityControlOption = 0 or 1 or 2') # or 3 or 4
    

In [ ]:
# EXECUTE THE FUNCTION
# CHAD PARALLELISED VERSION

# Parse time range once
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour,   EndMin   = int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay   = EndHour   * 60 + EndMin

# Shared QC kwargs — avoids repeating them in every call
qc_kwargs = dict(
    RadarIDno=RadarIDno, RadarYear=RadarYear, RadarMonth=RadarMonth,
    LoopStartTime=LoopStartTime, LoopEndTime=LoopEndTime,
    QualityControlOption=QualityControlOption,
    VarianceGridSize=VarianceGridSize,
    MinZ=MinZ, MinRhoHV=MinRhoHV,
    MaxZvariance=MaxZvariance, MaxVvariance=MaxVvariance,
    MaxZDRvariance=MaxZDRvariance, MaxRhoHVvariance=MaxRhoHVvariance,
    MinZcount=MinZcount, MinVcount=MinVcount,
    MinZDRcount=MinZDRcount, MinRhoHVcount=MinRhoHVcount,
    MinKDPcount=MinKDPcount,
)

# Build the list of delayed tasks
tasks_args = [
    (RadarDay, MinOfDay)
    for RadarDay in range(2, 28 + 1)
    for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5)
]

print(f'Total tasks queued: {len(tasks_args)}')




# LATEST OPTION WITH PROGRESS MONITORING
# n_workers should be 1 less than the number of cores (11 for LARGE ARE SESSION (12 cores))
cluster = LocalCluster(n_workers=11, threads_per_worker=1)
client  = Client(cluster)

# Submit all tasks
futures = [client.submit(QCMaskRadarData, day, t, **qc_kwargs)
           for day, t in tasks_args]

# Live progress bar in the notebook
for future in tqdm(as_completed(futures), total=len(futures), desc='QC Progress'):
    pass

# Collect results when done
results = client.gather(futures)
client.close()


# OLD OPTIONS WITHOUT PROGRESS MONITORING
# --- Option A: Local multiprocessing (simple, good for a single node) ---
# results = dask.compute(*tasks, scheduler='processes', num_workers=8)

# --- Option B: Distributed (better for NCI multi-node jobs) ---
# from dask.distributed import Client
# client = Client(n_workers=16, threads_per_worker=1)
# results = dask.compute(*tasks)
# client.close()

# OPTIONS
# # Threads — fast to start, but Python GIL limits true parallelism for CPU work
# dask.compute(*tasks, scheduler='threads', num_workers=8)

# # Processes — true parallelism, best for CPU-bound work like yours
# dask.compute(*tasks, scheduler='processes', num_workers=8)

# # Synchronous — single-threaded, useful for debugging
# dask.compute(*tasks, scheduler='synchronous')